# Unconditional Diffusion — Pokémon / Fusion Sprites

Feasibility check: can a diffusion model generate plausible random sprites?

- **Data**: `images/fusiondex/{base,fusions}/<...>/<id>.png`, scraped by `scrape_fusiondex.py`.
- **Model**: HuggingFace `diffusers` `UNet2DModel` + DDPM (train) / DDIM (sample).
- **Channels**: sprites are RGBA with *binary* alpha; we composite onto **black**
  (`rgb * alpha`) and train on **3 channels** (lossless for the foreground).

Set `cfg.max_images` small + `cfg.image_size=32` for a local CPU/MPS smoke test;
scale up on the DGX H100s. Training is resumable from the latest checkpoint.

In [ ]:
import os, glob, math, random, contextlib
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Reduce CUDA fragmentation OOMs on shared GPUs (must be set before importing torch)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import torch.nn.functional as F
from torchvision.utils import make_grid, save_image

from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_cosine_schedule_with_warmup

print("torch", torch.__version__)

In [ ]:
from dataclasses import dataclass, field
from typing import Tuple, Optional

@dataclass
class Config:
    data_root: str = "/data/cs282/pkmn-dataset"
    categories: Tuple[str, ...] = ("base", "fusions")  # subfolders to include
    max_images: Optional[int] = None   # set small (e.g. 256) for a smoke test

    image_size: int = 64               # 32 = fast smoke test, 96 = native, 128 = sharper
    channels: int = 3                  # RGB on black background

    batch_size: int = 32               # per-step batch; lower to 16/8 if you OOM
    grad_accum: int = 4                # effective batch = batch_size * grad_accum
    num_workers: int = 8
    lr: float = 1e-4
    num_epochs: int = 200
    warmup_steps: int = 500
    num_train_timesteps: int = 1000
    ema_decay: float = 0.9999

    sample_every: int = 10             # epochs between sample grids / checkpoints
    keep_last_ckpts: int = 3           # prune older checkpoints (each is ~1GB)
    n_sample: int = 16
    sample_steps: int = 50             # DDIM steps at inference

    ckpt_dir: str = "checkpoints/diffusion"
    sample_dir: str = "samples/diffusion"
    seed: int = 0

cfg = Config()

# --- smoke-test overrides: uncomment locally to verify the pipeline ---
# cfg.max_images = 256
# cfg.image_size = 32
# cfg.batch_size = 16
# cfg.num_workers = 0
# cfg.num_epochs = 2
# cfg.sample_every = 1

random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)

def pick_device():
    """Pick the CUDA GPU with the most free memory (respects CUDA_VISIBLE_DEVICES).

    Note: this is single-GPU. On a shared box, prefer pinning the job explicitly,
    e.g. launch with `CUDA_VISIBLE_DEVICES=2 jupyter lab`, which makes only that
    card visible and selectable here.
    """
    if torch.cuda.is_available():
        best, best_free = 0, -1
        for i in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(i)
            gib = 1024 ** 3
            print(f"  cuda:{i}  {free/gib:5.1f} GiB free / {total/gib:.0f} GiB")
            if free > best_free:
                best, best_free = i, free
        print(f"selected cuda:{best} ({best_free/1024**3:.1f} GiB free)")
        print("  NOTE: snapshot only -- other users may grab memory right after; "
              "if you OOM, lower cfg.batch_size or pin a GPU with CUDA_VISIBLE_DEVICES")
        return torch.device(f"cuda:{best}")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()

# bf16 autocast only pays off (and is only enabled) on CUDA / H100s
amp_dtype = torch.bfloat16
def amp_ctx():
    if device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=amp_dtype)
    return contextlib.nullcontext()

os.makedirs(cfg.ckpt_dir, exist_ok=True)
os.makedirs(cfg.sample_dir, exist_ok=True)
print("device:", device)

In [ ]:
class SpriteDataset(torch.utils.data.Dataset):
    """RGBA sprites composited onto black, downscaled, normalized to [-1, 1]."""

    def __init__(self, cfg: Config):
        paths = []
        for cat in cfg.categories:
            paths += glob.glob(os.path.join(cfg.data_root, cat, "**", "*.png"),
                               recursive=True)
        paths = sorted(paths)
        if cfg.max_images is not None and len(paths) > cfg.max_images:
            rng = random.Random(cfg.seed)
            paths = rng.sample(paths, cfg.max_images)
        if not paths:
            raise RuntimeError(f"No PNGs found under {cfg.data_root}/{cfg.categories}")
        self.paths = paths
        self.size = cfg.image_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGBA")
        arr = np.asarray(img, dtype=np.float32)
        rgb, alpha = arr[..., :3], arr[..., 3:4] / 255.0
        comp = (rgb * alpha).astype(np.uint8)              # transparent -> black
        comp = Image.fromarray(comp).resize(
            (self.size, self.size), Image.Resampling.BOX)  # area downscale
        t = torch.from_numpy(np.asarray(comp, dtype=np.float32) / 255.0)
        return t.permute(2, 0, 1) * 2.0 - 1.0              # [-1, 1], (3,H,W)

dataset = SpriteDataset(cfg)
print(f"{len(dataset)} sprites")

In [ ]:
loader = torch.utils.data.DataLoader(
    dataset, batch_size=cfg.batch_size, shuffle=True,
    num_workers=cfg.num_workers, pin_memory=(device.type == "cuda"),
    drop_last=True,
)

def show(batch, title=""):
    grid = make_grid(((batch[:cfg.n_sample] + 1) / 2).clamp(0, 1),
                     nrow=int(cfg.n_sample ** 0.5))
    plt.figure(figsize=(6, 6))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy()); plt.axis("off")
    plt.title(title); plt.show()

show(next(iter(loader)), "real samples (composited on black)")

In [ ]:
def build_unet(cfg: Config) -> UNet2DModel:
    # 4 levels -> 3 downsamples. For 96/128 px add a level, e.g.
    # block_out_channels=(128,256,256,512,512) with matching block tuples.
    return UNet2DModel(
        sample_size=cfg.image_size,
        in_channels=cfg.channels,
        out_channels=cfg.channels,
        layers_per_block=2,
        block_out_channels=(128, 256, 256, 512),
        down_block_types=("DownBlock2D", "DownBlock2D",
                          "AttnDownBlock2D", "DownBlock2D"),
        up_block_types=("UpBlock2D", "AttnUpBlock2D",
                        "UpBlock2D", "UpBlock2D"),
    )

model = build_unet(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"UNet2DModel: {n_params/1e6:.1f}M params")

In [ ]:
noise_scheduler = DDPMScheduler(
    num_train_timesteps=cfg.num_train_timesteps,
    beta_schedule="squaredcos_cap_v2",   # cosine: good for small images
)
ddim = DDIMScheduler(
    num_train_timesteps=cfg.num_train_timesteps,
    beta_schedule="squaredcos_cap_v2",
)

opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
ema = EMAModel(model.parameters(), decay=cfg.ema_decay, use_ema_warmup=True,
               model_cls=UNet2DModel, model_config=model.config)

steps_per_epoch = len(loader)
opt_steps = cfg.num_epochs * math.ceil(steps_per_epoch / cfg.grad_accum)
lr_sched = get_cosine_schedule_with_warmup(
    opt, num_warmup_steps=cfg.warmup_steps,
    num_training_steps=opt_steps,   # counts optimizer steps, not micro-batches
)
losses = []

In [ ]:
def save_checkpoint(epoch, step):
    path = os.path.join(cfg.ckpt_dir, f"ckpt_{epoch+1:04d}.pth")
    torch.save({
        "model": model.state_dict(),
        "ema": ema.state_dict(),
        "opt": opt.state_dict(),
        "lr_sched": lr_sched.state_dict(),
        "epoch": epoch, "step": step,
        "losses": losses,
        "config": vars(cfg),
    }, path)
    # prune old checkpoints to keep disk usage bounded (each is ~1GB)
    ckpts = sorted(glob.glob(os.path.join(cfg.ckpt_dir, "ckpt_*.pth")))
    for old in ckpts[:-cfg.keep_last_ckpts]:
        os.remove(old)
    return path

def maybe_resume():
    ckpts = sorted(glob.glob(os.path.join(cfg.ckpt_dir, "ckpt_*.pth")))
    if not ckpts:
        return 0, 0
    ck = torch.load(ckpts[-1], map_location=device)
    model.load_state_dict(ck["model"])
    ema.load_state_dict(ck["ema"])
    opt.load_state_dict(ck["opt"])
    lr_sched.load_state_dict(ck["lr_sched"])
    losses[:] = ck["losses"]
    print(f"resumed from {ckpts[-1]} @ epoch {ck['epoch']+1}")
    return ck["epoch"] + 1, ck["step"]

In [ ]:
@torch.no_grad()
def sample(n=None, steps=None, seed=None, noise=None):
    """Denoise to images. Pass `noise` to reuse fixed latents; else random
    (optionally `seed`-ed). Generator runs on CPU then moves -> device-portable."""
    steps = steps or cfg.sample_steps
    ddim.set_timesteps(steps)
    if noise is None:
        n = n or cfg.n_sample
        if seed is not None:
            g = torch.Generator().manual_seed(seed)
            noise = torch.randn(n, cfg.channels, cfg.image_size,
                                cfg.image_size, generator=g)
        else:
            noise = torch.randn(n, cfg.channels, cfg.image_size, cfg.image_size)
    x = noise.to(device).clone()

    ema.store(model.parameters()); ema.copy_to(model.parameters())
    model.eval()
    for t in ddim.timesteps:
        with amp_ctx():
            pred = model(x, t).sample
        x = ddim.step(pred.float(), t, x).prev_sample
    ema.restore(model.parameters())
    model.train()
    return ((x.clamp(-1, 1) + 1) / 2)

# Fixed latents: the per-epoch grids show the SAME seeds evolving, so you can
# read off real progress (the GAN notebook used `fixed_random` for this).
# Seeded from cfg.seed on CPU -> identical across runs/resumes and any device.
_g = torch.Generator().manual_seed(cfg.seed)
fixed_noise = torch.randn(cfg.n_sample, cfg.channels, cfg.image_size,
                          cfg.image_size, generator=_g).to(device)

def generate_grid(epoch):
    imgs = sample(noise=fixed_noise)
    grid = make_grid(imgs, nrow=int(cfg.n_sample ** 0.5))
    out = os.path.join(cfg.sample_dir, f"sample_epoch{epoch+1:04d}.png")
    save_image(grid, out)
    plt.figure(figsize=(6, 6))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy()); plt.axis("off")
    plt.title(f"epoch {epoch+1} (fixed seeds)"); plt.show()

In [ ]:
start_epoch, global_step = maybe_resume()
model.train()

for epoch in range(start_epoch, cfg.num_epochs):
    pbar = tqdm(loader, desc=f"epoch {epoch+1}/{cfg.num_epochs}")
    opt.zero_grad(set_to_none=True)
    for i, batch in enumerate(pbar):
        batch = batch.to(device)
        noise = torch.randn_like(batch)
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                          (batch.size(0),), device=device).long()
        noisy = noise_scheduler.add_noise(batch, noise, t)

        with amp_ctx():
            pred = model(noisy, t).sample
            loss = F.mse_loss(pred.float(), noise) / cfg.grad_accum
        loss.backward()

        if (i + 1) % cfg.grad_accum == 0:
            opt.step(); lr_sched.step(); ema.step(model.parameters())
            opt.zero_grad(set_to_none=True)

        full_loss = loss.item() * cfg.grad_accum
        losses.append(full_loss); global_step += 1
        pbar.set_postfix(loss=f"{full_loss:.4f}",
                         lr=f"{lr_sched.get_last_lr()[0]:.1e}")

    if (epoch + 1) % 2 == 0 or (epoch + 1) == cfg.num_epochs:
        generate_grid(epoch)

    if (epoch + 1) % cfg.sample_every == 0 or (epoch + 1) == cfg.num_epochs:
        print("saved", save_checkpoint(epoch, global_step))

## Inference — generate random sprites
Uses EMA weights and DDIM sampling. Re-run with different `seed` for new sprites.

In [ ]:
imgs = sample(cfg.n_sample, steps=cfg.sample_steps, seed=1234)
grid = make_grid(imgs, nrow=int(cfg.n_sample ** 0.5))
plt.figure(figsize=(7, 7))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy()); plt.axis("off")
plt.title("generated sprites (EMA)"); plt.show()

In [ ]:
if losses:
    plt.figure(figsize=(7, 3))
    plt.plot(losses, alpha=0.3, label="step")
    if len(losses) >= 50:
        k = 50
        smooth = np.convolve(losses, np.ones(k)/k, mode="valid")
        plt.plot(range(k-1, len(losses)), smooth, label=f"moving avg ({k})")
    plt.xlabel("step"); plt.ylabel("MSE loss"); plt.legend(); plt.show()

## Training progression
Stitch the per-epoch fixed-seed grids into a GIF to watch the *same* sprites
evolve from epoch 1 → N.

In [ ]:
frames = sorted(glob.glob(os.path.join(cfg.sample_dir, "sample_epoch*.png")))
if frames:
    imgs = [Image.open(f).convert("RGB") for f in frames]
    gif_path = os.path.join(cfg.sample_dir, "progression.gif")
    imgs[0].save(gif_path, save_all=True, append_images=imgs[1:],
                 duration=400, loop=0)
    print(f"wrote {gif_path} ({len(imgs)} frames)")
else:
    print("no sample grids yet -- train for at least one sample_every interval")

## Notes & next steps

- **Crisp pixels**: diffusion output is continuous. For a clean pixel-art look,
  nearest-neighbor downscale samples to the native 96×96 (or 48/64) after
  generation, optionally snapping to a small palette.
- **Background**: it's flat black, so it's trivially re-keyable to transparency
  (`alpha = any(rgb > threshold)`) if you want RGBA sprites back.
- **Feasibility bar**: with the full dataset you want to see blob→silhouette→
  creature-like structure emerge over epochs. If samples stay mushy, raise
  `image_size`/model capacity or train longer before considering the conditional
  (head+body → fusion) model.
- **Scaling on the DGX**: increase `batch_size`, `num_workers`, and `image_size`;
  bf16 autocast is already enabled on CUDA. Run inside `tmux`; training resumes
  from the latest checkpoint automatically.